# Xử lý dataset NVTS
1. Loại bỏ các class có < 20 ảnh
2. Crop ảnh dựa trên bounding box
3. Chia dataset thành 80/10/10 (Train/Val/Test)

In [5]:
import os
import cv2
from collections import defaultdict
import random
from tqdm import tqdm

# Đường dẫn dataset ban đầu
dataset_dir = "dataset_NVTS"
images_dir = os.path.join(dataset_dir, "images")
labels_dir = os.path.join(dataset_dir, "labels")
classes_file = os.path.join(dataset_dir, "classes.txt")

# Thư mục lưu dataset mới
output_dir = "cropped_dataset"
train_dir = os.path.join(output_dir, "train")
val_dir = os.path.join(output_dir, "val")
test_dir = os.path.join(output_dir, "test")

# Đọc danh sách classes
with open(classes_file, "r") as f:
    classes = [line.strip() for line in f.readlines()]

print(f"Tổng số classes ban đầu: {len(classes)}")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_NVTS/classes.txt'

In [ ]:
# Đếm số lượng ảnh cho mỗi class để lọc các class >= 20 ảnh
class_image_counts = defaultdict(int)
for filename in tqdm(os.listdir(labels_dir), desc="Counting classes"):
    if not filename.endswith(".txt"): continue
    filepath = os.path.join(labels_dir, filename)
    try:
        with open(filepath, "r") as f:
            class_ids = set()
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_ids.add(int(parts[0]))
            for cid in class_ids:
                class_image_counts[cid] += 1
    except Exception:
        pass

# Lọc các class có >= 20 ảnh
valid_classes_idx = {cid for cid, count in class_image_counts.items() if count >= 20}
valid_classes_names = {cid: classes[cid] for cid in valid_classes_idx}

print(f"Số lượng class được giữ lại (>= 20 ảnh): {len(valid_classes_idx)}")

In [ ]:
# Thu thập tất cả bounding boxes cho các class hợp lệ
class_crops = defaultdict(list)

for filename in tqdm(os.listdir(labels_dir), desc="Parsing labels"):
    if not filename.endswith(".txt"): continue
    img_filename = filename.replace(".txt", ".jpg")
    if not os.path.exists(os.path.join(images_dir, img_filename)): continue
    
    with open(os.path.join(labels_dir, filename), "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(parts[0])
                if cid in valid_classes_idx:
                    # Lưu thông tin crop: (tên_ảnh, x_center, y_center, width, height)
                    class_crops[cid].append((img_filename, float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])))

print("Đã thu thập xong danh sách bbox.")

In [ ]:
# Thực hiện crop và chia 80/10/10
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

for cid, crops in tqdm(class_crops.items(), desc="Cropping images"):
    random.seed(42)  # Để đảm bảo chia split giống nhau mỗi lần chạy
    random.shuffle(crops)
    
    n = len(crops)
    train_idx = int(n * 0.8)
    val_idx = int(n * 0.9)
    
    train_crops = crops[:train_idx]
    val_crops = crops[train_idx:val_idx]
    test_crops = crops[val_idx:]
    
    # Đảm bảo tên thư mục an toàn
    class_name_safe = valid_classes_names[cid].replace("/", "_").replace("*", "_")
    os.makedirs(os.path.join(train_dir, class_name_safe), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_name_safe), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name_safe), exist_ok=True)
    
    def process_crops(crop_list, split_name):
        for i, (img_name, x, y, w, h) in enumerate(crop_list):
            img_path = os.path.join(images_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            
            H, W, _ = img.shape
            x_min = int((x - w/2) * W)
            y_min = int((y - h/2) * H)
            x_max = int((x + w/2) * W)
            y_max = int((y + h/2) * H)
            
            # Cắt ảnh theo biên an toàn
            x_min, y_min = max(0, x_min), max(0, y_min)
            x_max, y_max = min(W, x_max), min(H, y_max)
            
            if x_max > x_min and y_max > y_min:
                cropped = img[y_min:y_max, x_min:x_max]
                out_name = f"{img_name.split('.')[0]}_{i}.jpg"
                out_path = os.path.join(output_dir, split_name, class_name_safe, out_name)
                cv2.imwrite(out_path, cropped)

    process_crops(train_crops, "train")
    process_crops(val_crops, "val")
    process_crops(test_crops, "test")

print("Hoàn tất tạo dataset mới với tập Test!")